# HW2：Test-time Steering 的回顧與延伸

<div style="background:rgba(255,183,0,.16); border-left:6px solid #e8a317; border-radius:4px; padding:8px 12px; margin-top:6px;">這一份只有黃色 block 下面的 cell 要填寫。</div>
<div style="background:rgba(42,120,214,.10); border-left:6px solid #2a78d6; border-radius:4px; padding:8px 12px; margin-top:6px; margin-bottom:14px;">藍色 block 是選填的部分。</div>

這次的作業主要是延伸上次的 lab（多跑一個高維測資＋寫下你的想法）。

首先，先把你在 lab（test time steering 大挑戰）寫的方法貼到下面的 `my_method`，再從頭跑到尾就可以了。

- Part A 是上課時一起進行過的 1D 題目。這次的 code 會把你 sample 出來的點，和真正的 target distribution $p_R$ 畫在一起做比較。
- Part B 換成另一個訓練在更高維度的資料上的模型。code 會用你的 `my_method` 再跑一次 test time steering。

報告要回答的問題在 `README.md`。

規則跟 lab 一樣：模型參數不能動；交出去的點要是模型 sample 出來的（呼叫模型請用 `v_theta` 或 `sample`，NFE 才會算到）；每次產生 2,000 個點，NFE 不超過 1,000,000。

In [ ]:
#@title 下載檔案、載入模型（不用修改，直接 run） { display-mode: "form" }
import math, os, urllib.request
import numpy as np, pandas as pd, torch, torch.nn as nn
import matplotlib.pyplot as plt

REPO = "https://raw.githubusercontent.com/shihhsinwang0214/dma-hw2/main/"
for f in ["fm_model.pt", "target_1d.csv", "fm_model_B.pt", "reference_B.npz"]:
    if not os.path.exists(f):
        print("downloading", f); urllib.request.urlretrieve(REPO + f, f)

class TimeEmb(nn.Module):
    def __init__(self, dim=64):
        super().__init__(); self.dim = dim
    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(1000.0) * torch.arange(half, dtype=torch.float32) / half)
        ang = t[:, None] * 1000.0 * freqs[None, :]
        return torch.cat([ang.sin(), ang.cos()], 1)

class FlowNet(nn.Module):
    def __init__(self, d=1, hidden=256, temb=64):
        super().__init__()
        self.temb = TimeEmb(temb)
        self.net = nn.Sequential(nn.Linear(d + temb, hidden), nn.SiLU(),
                                 nn.Linear(hidden, hidden), nn.SiLU(),
                                 nn.Linear(hidden, hidden), nn.SiLU(),
                                 nn.Linear(hidden, d))
    def forward(self, x, t):
        return self.net(torch.cat([x, self.temb(t)], 1))

def _load(path, d, hidden):
    m = FlowNet(d, hidden).eval()
    m.load_state_dict(torch.load(path, map_location="cpu"))
    for p in m.parameters():
        p.requires_grad_(False)
    return m

_B = np.load("reference_B.npz")
D_B, LAM_B, S2_B = int(_B["D"]), float(_B["LAM"]), float(_B["S2"])
XSTAR = torch.tensor(_B["XSTAR"])
MODEL_A = _load("fm_model.pt", 1, 256)
MODEL_B = _load("fm_model_B.pt", D_B, 512)
print("models loaded")

## Lab 裡用過的函式

函式的名字和用法都跟 lab 的 starter 一樣，所以 lab 的程式可以直接貼過來用。差別只有點的形狀改成 `(n, D)`：Part A 的 `D = 1`，Part B 的 `D` 比較大。用 `use("A")`、`use("B")` 切換題目，`model`、`reward`、`LAM`、`D` 都會跟著換。

| 名字 | 意思 | 會算進 NFE 嗎 |
|---|---|---|
| `v_theta(x_t, t)` | velocity $v_\theta(x_t,t)$，輸入輸出都是 `(n,D)` | 會（+n） |
| `x_hat(x_t, t, v)` | 估計 clean data $\hat x = x_t + (1-t)\,v_\theta(x_t, t)$ | 不會 |
| `sample(n, steps, z=None)` | 最基本的 Euler sampling，回傳 numpy `(n, D)`；可以把裡面的迴圈複製出來改 | 會（n×steps） |
| `reward(x)` | 這一題的 reward，輸入 `(n,D)`、輸出 `(n,)`；輸入 tensor 的話可以算 gradient | 不會 |
| `LAM`、`D` | 這一題的 λ 和維度 | — |
| `NFE` | 目前累計的網路呼叫次數 | — |
| `compare(x)` | 把你的 sample 和 $p_R$ 畫在一起，並印出數字（lab 的 `show` 也還能用） | 不會 |

In [ ]:
NFE = 0

def v_theta(x_t, t):
    """x_t: (n,D)；t: float 或 (n,) tensor。要對 x_t 求 gradient 就傳 requires_grad 的 x_t。"""
    global NFE
    x_t = torch.as_tensor(x_t, dtype=torch.float32).reshape(-1, D)
    tt = torch.full((x_t.shape[0],), float(t)) if np.ndim(t) == 0 else torch.as_tensor(t, dtype=torch.float32).reshape(-1)
    NFE += x_t.shape[0]
    return model(x_t, tt)

def x_hat(x_t, t, v):
    return x_t + (1 - t) * v

@torch.no_grad()
def sample(n=2000, steps=100, z=None):
    x = torch.randn(n, D) if z is None else torch.as_tensor(z, dtype=torch.float32).reshape(-1, D).clone()
    ts = torch.linspace(0, 1, steps + 1)
    for i in range(steps):
        t, dt = float(ts[i]), float(ts[i + 1] - ts[i])
        v = v_theta(x, t)            # 算 NFE
        xh = x_hat(x, t, v)          # 估計 clean data，不另外花 NFE
        x = x + dt * v               # Euler 一步
    return x.numpy()

PEAKS_A = torch.tensor([[-7.2, 2.0], [-4.2, 4.0], [-1.3, 3.0], [4.3, 5.0], [7.2, 1.0]])   # Part A 的 reward 峰：(中心, 高度)，sd 都是 0.3

def reward(x):
    """numpy 或 tensor，形狀 (n,D)（Part A 也接受 (n,)）→ (n,)，型別和輸入一樣。"""
    is_np = not isinstance(x, torch.Tensor)
    xt = torch.tensor(np.asarray(x, dtype=np.float32)) if is_np else x.float()
    xt = xt.reshape(-1, D)
    if PROBLEM == "A":
        r = (PEAKS_A[:, 1] * torch.exp(-(xt - PEAKS_A[:, 0]) ** 2 / (2 * 0.3 ** 2))).sum(1)
    else:
        r = -((xt - XSTAR) ** 2).sum(1) / (2 * S2_B)
    return r.numpy().astype(float) if is_np else r

def use(problem):
    """切換題目：use("A") 是 lab 的 1D 題目，use("B") 是 Part B 的模型。"""
    global PROBLEM, model, LAM, D, PEAKS
    if problem == "A":
        model, LAM, D, PEAKS = MODEL_A, 3.0, 1, PEAKS_A
    elif problem == "B":
        model, LAM, D, PEAKS = MODEL_B, LAM_B, D_B, None
    else:
        raise ValueError('請用 use("A") 或 use("B")')
    PROBLEM = problem
    print(f"Part {problem}：D = {D}，LAM = {LAM}")

use("A")

## 評估流程（evaluation pipeline）

這一個 cell 不用修改，直接 run 就好。

- `compare(x, title)` 會畫圖，並印出一次結果的比較表。
- `run(method)` 會用 `N_SEEDS` 個 random seed 各跑一次 `method()`，把每個 seed 的結果畫在一起，再印一張總表：每個 metric 的平均、95% 信賴區間、幾次超出範圍、差幾倍。回傳 `(每個 seed 的數字, 每個 seed 的 sample)`。

比較表裡用的 metric，是從大家 HW1 A 中形成的五種類別（HW1 A 分組討論時便利貼的五個顏色）各挑一兩個：藍 搬運距離、粉紅 成對距離、綠 密度與分格、黃 數點、橘 先分群再比。每個 metric 的範圍，是從 $p_R$ 抽 20 組、每組 2,000 個點，各算一次得到的最小到最大，跟 HW1 A reflection 中的「比真實互比都差」是同一個做法。

- **超出幾次**：`N_SEEDS` 次裡，有幾次落在範圍外，括號裡是超出的方向。就算真的從 $p_R$ 抽，每個 metric 單看一次也有大約 5–10% 的機會剛好落在範圍外，所以每一次都超出才標紅。哪一邊算超出，每個 metric 不一樣。
- **差幾倍**：每個 seed 各算一次比範圍差了幾倍（沒超出算 1），再取中位數（一半以上的 seed 沒超出，就會寫「沒超出」）。倍數怎麼算，每個 metric 也不一樣。
- 「不同的點」只當參考，不標紅。

每個 metric 的定義、哪一邊算超出、差幾倍怎麼算、看不到什麼，寫在這一格下面的「比較表的 metric：定義與說明」。

In [ ]:
#@title 評估流程（不用修改，直接 run） { display-mode: "form" }
import numpy as np, torch

# HW1 A 的五個便利貼顏色（每一類的 metric 在 HW1 共同測試裡看得出、看不出的東西不一樣）
FAMILY = {
    "藍": "藍 搬運距離",
    "粉紅": "粉紅 成對距離",
    "綠": "綠 密度與分格",
    "黃": "黃 數點",
    "橘": "橘 先分群再比",
}

# 超出時的方向要怎麼讀（沒有列出來的 metric 就只寫「偏大」或「偏小」）
HINT = {
    "coverage": {"偏大": "太集中", "偏小": "有地方沒蓋到"},
    "width": {"偏大": "太散", "偏小": "太窄"},
    "width_in": {"偏大": "太散", "偏小": "太窄"},
    "width_out": {"偏大": "太散", "偏小": "太窄"},
}


def _w1_1d(a, b):
    a = np.sort(a.reshape(-1)); b = np.sort(b.reshape(-1)); g = np.sort(np.concatenate([a, b]))
    fa = np.searchsorted(a, g, side="right") / len(a); fb = np.searchsorted(b, g, side="right") / len(b)
    return float(np.sum(np.abs(fa - fb)[:-1] * np.diff(g)))


class Panel:
    """同一份參考資料（從 p_R 抽的點）上算一整排 metric，並用 p_R 自己的幾組樣本量出「沒有錯時」的範圍。

    ref: (m, D) 參考點；centers: 分群用的中心（D=1 時是位置，D>1 時是投影平面上的座標）；
    project: D>1 時把點投影到平面的矩陣 (2, D)，兩列互相垂直、長度 1；h: MMD 的 kernel 寬度；
    null_sets: 好幾組從 p_R 抽的 (n, D)。
    """

    def __init__(self, ref, centers, h, null_sets, project=None, tv_edges=None, k=5, n_dir=128):
        self.ref = np.asarray(ref, dtype=np.float64)
        self.D = self.ref.shape[1]
        self.centers = np.asarray(centers, dtype=np.float64)
        self.project, self.h, self.tv_edges, self.k = project, h, tv_edges, k
        R = torch.as_tensor(self.ref)
        self.R = R
        d = torch.cdist(R, R)
        m = len(R)
        self.dyy = float(d.sum() / (m * (m - 1)))
        kr = torch.exp(-d ** 2 / (2 * h * h))
        self.kyy = float((kr.sum() - m) / (m * (m - 1)))
        self.radius = d.kthvalue(k + 1, dim=1).values            # 每個參考點到第 k 個鄰居的距離（不含自己）
        self.lr = self.label(self.ref)
        self.K = len(self.centers)
        self.main = [g for g in range(self.K) if np.mean(self.lr == g) > 0.05]
        self.ref_hist = self._hist(self.ref)
        self.ref_width = {g: self._spread(self.ref[self.lr == g]) for g in self.main}
        if self.D > 1:
            u = np.random.default_rng(0).standard_normal((n_dir, self.D))
            self.dirs = u / np.linalg.norm(u, axis=1, keepdims=True)
            self.ref_proj = self.ref @ self.dirs.T
        self.null_sets = [np.asarray(z, dtype=np.float64).reshape(-1, self.D) for z in null_sets]
        self.null = [self.scores(z) for z in self.null_sets]

    def null_range(self, f):
        """把任何 f(x, ref) -> float 用在 p_R 自己的那幾組樣本上，回傳 (最小, 最大)。"""
        v = [float(f(z, self.ref)) for z in self.null_sets]
        return min(v), max(v)

    def label(self, x):
        p = x if self.D == 1 else x @ self.project.T
        c = self.centers.reshape(len(self.centers), -1)
        return np.argmin(((p[:, None, :] - c[None]) ** 2).sum(-1), axis=1)

    def _hist(self, x):
        """分格的點數比例，最後多一格放「格子外面」的點。"""
        if self.D == 1:
            a, _ = np.histogram(x.reshape(-1), self.tv_edges[0])
        else:
            p = x @ self.project.T
            a, _, _ = np.histogram2d(p[:, 0], p[:, 1], self.tv_edges)
        a = a.reshape(-1) / len(x)
        return np.append(a, 1 - a.sum())

    def _spread(self, a):
        """一群點散開的程度：每個座標的變異數取平均再開根號。D>1 時分成平面上（2 個方向）和平面外（其餘 D−2 個方向）。"""
        if len(a) < 2:
            return (0.0, 0.0) if self.D > 1 else 0.0
        if self.D == 1:
            return float(np.sqrt(a.var()))
        v_all = a.var(axis=0).sum()
        v_in = (a @ self.project.T).var(axis=0).sum()
        return float(np.sqrt(v_in / 2)), float(np.sqrt(max(v_all - v_in, 0) / (self.D - 2)))

    def scores(self, x):
        x = np.asarray(x, dtype=np.float64).reshape(-1, self.D)
        X = torch.as_tensor(x)
        n = len(X)
        dxy = torch.cdist(X, self.R)
        dxx = torch.cdist(X, X)
        out = {}
        # 藍 搬運距離：W1（D>1 時是 128 個固定隨機方向上的一維 W1 的平均）
        if self.D == 1:
            out["transport"] = _w1_1d(x, self.ref)
        else:
            px = x @ self.dirs.T
            out["transport"] = float(np.mean([_w1_1d(px[:, j], self.ref_proj[:, j]) for j in range(px.shape[1])]))
        # 粉紅 成對距離：energy distance（U-statistic，同分布時期望值是 0）
        out["energy"] = float(2 * dxy.mean() - dxx.sum() / (n * (n - 1)) - self.dyy)
        # 綠 密度與分格：MMD²（RBF kernel，U-statistic）與分格 TV（格子外面的點算一格）
        kxx = torch.exp(-dxx ** 2 / (2 * self.h ** 2))
        out["mmd"] = float((kxx.sum() - n) / (n * (n - 1)) + self.kyy - 2 * torch.exp(-dxy ** 2 / (2 * self.h ** 2)).mean())
        out["tv"] = float(0.5 * np.abs(self._hist(x) - self.ref_hist).sum())
        # 黃 數點：各群比例差與覆蓋率
        lx = self.label(x)
        out["ratio"] = float(np.abs(np.bincount(lx, minlength=self.K) / n - np.bincount(self.lr, minlength=self.K) / len(self.lr)).sum())
        out["coverage"] = float((dxy.min(dim=0).values <= self.radius).double().mean())
        # 橘 先分群再比：主要幾群的寬度比（1 是剛好），用參考資料裡各群的比例加權平均
        w = {g: np.mean(self.lr == g) for g in self.main}
        if self.D == 1:
            out["width"] = float(sum(w[g] * self._spread(x[lx == g]) / self.ref_width[g] for g in self.main) / sum(w.values()))
        else:
            s = {g: self._spread(x[lx == g]) for g in self.main}
            out["width_in"] = float(sum(w[g] * s[g][0] / self.ref_width[g][0] for g in self.main) / sum(w.values()))
            out["width_out"] = float(sum(w[g] * s[g][1] / self.ref_width[g][1] for g in self.main) / sum(w.values()))
        out["distinct"] = int(len(np.unique(x.round(6), axis=0)))
        return out

    def spec(self):
        """(key, 類別, 名稱, 哪一邊算超出)。
        big：偏大才算；big2：偏大才算，而且是平方的量（差幾倍開根號再比）；cov：兩邊都算（覆蓋率）；
        both：兩邊都算（寬度比，1 是剛好）；two：兩邊都算（自己加的 metric）。"""
        out = [
            ("transport", FAMILY["藍"], "W1" if self.D == 1 else "sliced W1（128 個方向）", "big"),
            ("energy", FAMILY["粉紅"], "energy distance", "big2"),
            ("mmd", FAMILY["綠"], f"MMD²（RBF，h = {self.h:g}）", "big2"),
            ("tv", FAMILY["綠"], "分格 TV（寬 0.1 的格子）" if self.D == 1 else "分格 TV（投影平面上 16×16 格）", "big"),
            ("ratio", FAMILY["黃"], "各群比例差", "big"),
            ("coverage", FAMILY["黃"], f"覆蓋率（coverage，k = {self.k}）", "cov"),
        ]
        if self.D == 1:
            out.append(("width", FAMILY["橘"], "各群寬度比（1 是剛好）", "both"))
        else:
            out.append(("width_in", FAMILY["橘"], "各群寬度比：平面上（1 是剛好）", "both"))
            out.append(("width_out", FAMILY["橘"], "各群寬度比：平面外（1 是剛好）", "both"))
        return out

    def range_of(self, key):
        v = [z[key] for z in self.null]
        return min(v), max(v)

    @staticmethod
    def side(v, lo, hi, way):
        """落在範圍外的方向：「偏大」、「偏小」或空字串。"""
        if way in ("big", "big2", "both", "two", "cov") and v > hi:
            return "偏大"
        if way in ("small", "both", "two", "cov") and v < lo:
            return "偏小"
        return ""

    @staticmethod
    def say(key, sd):
        """把方向寫成看得懂的字，例如覆蓋率偏大＝「偏大：太集中」。"""
        if not sd:
            return ""
        h = HINT.get(key, {}).get(sd)
        return f"{sd}：{h}" if h else sd

    @staticmethod
    def times(v, lo, hi, way):
        """比 p_R 自己的範圍差幾倍：沒超出＝1，超出時一定大於 1。
        big：值 ÷ 範圍上限；big2：√值 ÷ √範圍上限；
        cov：偏小時 (1−值) ÷ (1−範圍下限)，偏大時 (1−範圍上限) ÷ (1−值)；
        both：|log 值| ÷ 同一側那一端的 |log|（偏窄和下限比、偏寬和上限比）；
        two（和其他情況）：離範圍中點的距離 ÷ 範圍的半寬。"""
        if not Panel.side(v, lo, hi, way):
            return 1.0
        if way == "big" and hi > 0:
            return float(v / hi)
        if way == "big2" and hi > 0:
            return float(np.sqrt(v / hi))
        if way in ("small", "cov") and v < lo and lo < 1:
            return float((1 - v) / (1 - lo))
        if way == "cov" and v > hi:
            return float((1 - hi) / (1 - v)) if v < 1 else float("inf")
        if way == "both" and lo > 0:
            if v <= 0:
                return float("inf")
            if v < lo < 1:
                return float(np.log(v) / np.log(lo))
            if v > hi > 1:
                return float(np.log(v) / np.log(hi))
            a, b = np.log(lo), np.log(hi)
            return float(abs(np.log(v) - (a + b) / 2) / max((b - a) / 2, 1e-12))
        mid, half = (lo + hi) / 2, max((hi - lo) / 2, 1e-12)
        return float(abs(v - mid) / half)

    def rows(self, x, extra=None):
        """一次的結果：(名稱, 類別, 你的值, 範圍下限, 範圍上限, 判斷, 差幾倍) 的列表，以及所有分數。
        extra: {名稱: (類別, 函式 f(x, ref), 範圍)}。最後一列「不同的點」只當參考，不算超出。"""
        s = self.scores(x)
        n = len(np.asarray(x).reshape(-1, self.D))
        out = []
        for key, fam, name, way in self.spec():
            lo, hi = self.range_of(key)
            v = s[key]
            sd = self.side(v, lo, hi, way)
            out.append((name, fam, v, lo, hi, f"超出（{self.say(key, sd)}）" if sd else "", self.times(v, lo, hi, way)))
        for name, (fam, f, (lo, hi)) in (extra or {}).items():
            v = float(f(np.asarray(x, dtype=np.float64).reshape(-1, self.D), self.ref))
            s[name] = v
            sd = self.side(v, lo, hi, "two")
            out.append((name, fam, v, lo, hi, f"超出（{sd}）" if sd else "", self.times(v, lo, hi, "two")))
        # 一維時，真的從 p_R 抽的 2,000 個點四捨五入到小數 6 位後也會有幾個撞在一起，所以和 p_R 自己的最少個數比
        lo_d = self.range_of("distinct")[0]
        out.append(("不同的點", "", s["distinct"], lo_d, n, "比 p_R 少（有重複的點）" if s["distinct"] < lo_d else "", None))
        return out, s

    # 每個 metric 可能的最小、最大值（信賴區間不會超出這裡）；energy 和 MMD² 用的是 U-statistic，可以是負的
    BOUNDS = {"transport": (0, None), "energy": (None, None), "mmd": (None, None), "tv": (0, 1),
              "ratio": (0, 2), "coverage": (0, 1), "width": (0, None), "width_in": (0, None), "width_out": (0, None),
              "distinct": (0, None)}

    def summary(self, xs, extra=None):
        """好幾個 seed 一起看：每個 metric 的平均、95% 信賴區間（平均 ± t·標準誤，截在 metric 可能的範圍內）、
        幾次超出範圍（和超出時多數的方向）、差幾倍（每個 seed 各算一次，取中位數）。
        xs: 每個 seed 的樣本（list）。回傳 (每個 metric 一個 dict 的列表, 每個 seed 的分數)。"""
        per = [self.scores(x) for x in xs]
        N = len(per)
        tq = _t975(N - 1) if N > 1 else None

        def ci(vals, key=None):
            v = np.asarray(vals, dtype=float)
            m = float(v.mean())
            if N < 2:
                return m, None, None
            h = tq * v.std(ddof=1) / np.sqrt(N)
            a, b = m - h, m + h
            lo, hi = self.BOUNDS.get(key, (None, None))
            return m, (a if lo is None else max(a, lo)), (b if hi is None else min(b, hi))

        out = []
        for key, fam, name, way in self.spec():
            lo, hi = self.range_of(key)
            vals = [p[key] for p in per]
            m, a, b = ci(vals, key)
            r = dict(key=key, name=name, family=fam, mean=m, ci=(a, b), range=(lo, hi), n=N,
                     k=sum(1 for v in vals if self.side(v, lo, hi, way)), **_per_seed(vals, lo, hi, way))
            r["say"] = self.say(key, r["side"])
            out.append(r)
        for name, (fam, f, (lo, hi)) in (extra or {}).items():
            vals = [float(f(np.asarray(x, dtype=np.float64).reshape(-1, self.D), self.ref)) for x in xs]
            for p, v in zip(per, vals):
                p[name] = v
            m, a, b = ci(vals)
            r = dict(key=name, name=name, family=fam, mean=m, ci=(a, b), range=(lo, hi), n=N,
                     k=sum(1 for v in vals if self.side(v, lo, hi, "two")), **_per_seed(vals, lo, hi, "two"))
            r["say"] = r["side"]
            out.append(r)
        n = len(np.asarray(xs[0]).reshape(-1, self.D))
        dist = [p["distinct"] for p in per]
        m, a, b = ci(dist, "distinct")
        b = min(b, n) if b is not None else None
        lo_d = self.range_of("distinct")[0]
        out.append(dict(key="distinct", name="不同的點", family="", mean=m, ci=(a, b), range=(lo_d, n), n=N,
                        k=sum(1 for d in dist if d < lo_d), times=None, side="", say="", judged=False))
        return out, per


def _per_seed(vals, lo, hi, way):
    """每個 seed 各算一次差幾倍（沒超出算 1）取中位數；方向取超出的那幾次裡的多數。
    寬度比這種兩邊都算錯的 metric，各個 seed 可能一個偏寬一個偏窄，平均起來反而像是剛好，所以不用平均值來算。"""
    ts = [Panel.times(v, lo, hi, way) for v in vals]
    sides = [s for s in (Panel.side(v, lo, hi, way) for v in vals) if s]
    return dict(times=float(np.median(ts)), side=max(("偏大", "偏小"), key=sides.count) if sides else "", judged=True)


def _t975(df):
    """t 分布的 97.5% 分位數（算 95% 信賴區間用）。"""
    try:
        from scipy.stats import t
        return float(t.ppf(0.975, df))
    except Exception:
        table = {1: 12.706, 2: 4.303, 3: 3.182, 4: 2.776, 5: 2.571, 6: 2.447, 7: 2.365, 8: 2.306, 9: 2.262,
                 10: 2.228, 15: 2.131, 20: 2.086, 30: 2.042}
        if df > 30:
            return 1.96
        return table[min(table, key=lambda d: abs(d - df))]

def _np(x):
    return x.detach().cpu().numpy() if isinstance(x, torch.Tensor) else np.asarray(x)

_T = pd.read_csv("target_1d.csv"); _GX, _GP = _T["x"].to_numpy(), _T["p_R"].to_numpy()
_cdf = np.cumsum(_GP); _cdf /= _cdf[-1]
_inv = lambda u: np.interp(u, _cdf, _GX)
_REF_A = _inv((np.arange(4000) + 0.5) / 4000)[:, None]      # p_R 的 4,000 個 quantile（和 Kaggle 計分用的同一種）
_rng = np.random.default_rng(2026)
PANEL_A = Panel(_REF_A, [-7, -4, -1, 1.5, 4, 7], 0.2, [_inv(_rng.random(2000))[:, None] for _ in range(20)],
                tv_edges=[np.arange(-9, 9.0001, 0.1)])
BASIS = _B["BASIS"].astype(float)                            # Part B 畫圖用的投影平面（2 個方向）
_REF_B = _B["X_REF"].astype(float)
PANEL_B = Panel(_REF_B, _B["CENTERS"], 2.0, _B["X_NULL"], project=BASIS, tv_edges=[np.linspace(-4, 4, 17)] * 2)
_EXTRA = {"A": {}, "B": {}}
pd.set_option("display.unicode.east_asian_width", True)
print("比較表準備好了")

def add_metric(name, f, family="自己加的"):
    """把自己的 metric 加進比較表。f(x, ref) -> 一個數字；x、ref 都是 (n, D) 的 numpy array。"""
    for prob, P in [("A", PANEL_A), ("B", PANEL_B)]:
        try:
            _EXTRA[prob][name] = (family, f, P.null_range(f))
            print(f"Part {prob}：加入「{name}」，p_R 自己 20 次的範圍 {_EXTRA[prob][name][2][0]:.4g} ～ {_EXTRA[prob][name][2][1]:.4g}")
        except Exception as e:
            print(f"Part {prob}：「{name}」算不出來（{type(e).__name__}: {e}）")

_SHORT = {"transport": "搬運距離", "energy": "energy", "mmd": "MMD²", "tv": "分格TV", "ratio": "比例差",
          "coverage": "覆蓋率", "width": "寬度比", "width_in": "平面上寬度比", "width_out": "平面外寬度比", "distinct": "不同的點"}
_f = lambda v: f"{v:.4g}"

def _times(t):
    if t is None:
        return "—"
    if t <= 1.0:
        return "沒超出"
    return f"{t:,.0f} 倍" if t >= 100 else f"{t:.2g} 倍"

def _show(tab, red=None):
    """在 notebook 裡畫成表格，red[i] 為 True 的那一列把判斷欄標紅；不在 notebook 裡就印文字。"""
    try:
        from IPython.display import display
        st = tab.style.hide(axis="index")
        if red is not None:
            cols = {"判斷", "超出幾次", "差幾倍"}
            st = st.apply(lambda row: ["color:#b23b2a;font-weight:600" if (red[row.name] and c in cols) else "" for c in row.index], axis=1)
        display(st)
    except Exception:
        print(tab.to_string(index=False))

_PB = _REF_B @ BASIS.T
_right = lambda p: float(np.mean((p[:, 0] > 1.0) & (np.abs(p[:, 1]) < 1.5)))   # 投影圖右邊那團
_up = lambda p: float(np.mean((p[:, 1] > 1.0) & (np.abs(p[:, 0]) < 1.5)))      # 投影圖上面那團
_near = lambda a, c: float(np.mean(np.abs(a.reshape(-1) - c) < 1.5))

def _groups(x):
    """兩個主要的群各占多少：Part A 是 x≈4、x≈-4；Part B 是投影圖上右邊、上面那兩團。"""
    if PROBLEM == "A":
        return [("x≈4 的比例", _near(x, 4), _near(_REF_A, 4)), ("x≈-4 的比例", _near(x, -4), _near(_REF_A, -4))]
    p = x @ BASIS.T
    return [("右邊那團", _right(p), _right(_PB)), ("上面那團", _up(p), _up(_PB))]

def _plot_row(axs, x, title):
    """畫一個 seed：Part A 一張直方圖；Part B 四張（投影 ×2、R(x)、離平面的距離）。"""
    if PROBLEM == "A":
        ax = axs[0]
        ax.hist(x.reshape(-1), bins=np.linspace(-9, 9, 361), density=True, color="#4cc9f0", alpha=.75, label="your samples")
        ax.plot(_GX, _GP, color="k", lw=1.2, label="true $p_R$")
        ax2 = ax.twinx(); ax2.plot(_GX, reward(_GX), color="#c2410c", lw=.8, alpha=.6); ax2.set_ylim(0, 5.6); ax2.set_yticks([])
        ax.set_xlim(-9, 9); ax.set_title(title); ax.legend(loc="upper left")
        return
    p = x @ BASIS.T
    off, off_r = np.linalg.norm(x - p @ BASIS, axis=1), np.linalg.norm(_REF_B - _PB @ BASIS, axis=1)
    r, r_ref = reward(x), reward(_REF_B)
    star = XSTAR.numpy() @ BASIS.T
    for a, q, nm, c in [(axs[0], p, f"{title}: your samples (projected)", "#4cc9f0"), (axs[1], _PB, "true $p_R$ (projected)", "k")]:
        a.scatter(q[:, 0], q[:, 1], s=3, alpha=.35, color=c); a.plot(*star, "*", color="#c2410c", ms=12)
        a.set_xlim(-4, 4); a.set_ylim(-4, 4); a.set_aspect("equal"); a.set_title(nm, fontsize=10)
    rb = np.linspace(min(r_ref.min(), np.percentile(r, 1)) - 1, 0, 60)
    axs[2].hist(r_ref, bins=rb, density=True, histtype="step", color="k", label="true $p_R$")
    axs[2].hist(r, bins=rb, density=True, alpha=.6, color="#4cc9f0", label="yours"); axs[2].set_title("reward R(x)", fontsize=10); axs[2].legend()
    ob = np.linspace(0, max(off.max(), off_r.max()) * 1.05, 60)
    axs[3].hist(off_r, bins=ob, density=True, histtype="step", color="k", label="true $p_R$")
    axs[3].hist(off, bins=ob, density=True, alpha=.6, color="#4cc9f0", label="yours"); axs[3].set_title("distance off the plane", fontsize=10); axs[3].legend()

def _figure(n_rows):
    if PROBLEM == "A":
        fig, ax = plt.subplots(n_rows, 1, figsize=(10, 2.8 * n_rows), squeeze=False)
    else:
        fig, ax = plt.subplots(n_rows, 4, figsize=(17, 3.6 * n_rows), squeeze=False)
    return fig, ax

def compare(x, title=""):
    """一次的結果：畫圖，並印出比較表（判斷只看這一次）。"""
    P = PANEL_A if PROBLEM == "A" else PANEL_B
    x = _np(x).astype(float).reshape(-1, D)
    rows, s = P.rows(x, _EXTRA[PROBLEM])
    print(f"{title}　NFE = {NFE:,}　mean R = {reward(x).mean():.2f}")
    tab = pd.DataFrame([(fam, name, _f(v) if isinstance(v, float) else f"{v}/{len(x)}",
                         f"{lo} ～ {hi}" if name == "不同的點" else f"{_f(lo)} ～ {_f(hi)}", flag, _times(t))
                        for name, fam, v, lo, hi, flag, t in rows],
                       columns=["HW1 的類別", "metric", "你的值", "p_R 自己 20 次的範圍", "判斷", "差幾倍"])
    _show(tab, red=[bool(r[5]) and r[0] != "不同的點" for r in rows])      # 不同的點只當參考，不標紅
    print("　".join(f"{nm} {v:.2f}（p_R 是 {ref:.2f}）" for nm, v, ref in _groups(x)))
    fig, ax = _figure(1); _plot_row(ax[0], x, title); plt.tight_layout(); plt.show()
    out = {_SHORT.get(k, k): v for k, v in s.items()}
    return out

show = compare      # lab 的名字

def run(method, n_seeds=None):
    """用 n_seeds 個 random seed（預設 N_SEEDS）各跑一次 method()，畫在一起，最後印總表。"""
    global NFE
    n_seeds = N_SEEDS if n_seeds is None else n_seeds
    P = PANEL_A if PROBLEM == "A" else PANEL_B
    xs, nfes = [], []
    for k in range(n_seeds):
        torch.manual_seed(k); np.random.seed(k)
        NFE = 0
        xs.append(_np(method()).astype(float).reshape(-1, D))
        nfes.append(NFE)
        if NFE > 1_000_000:
            print(f"注意：seed {k} 用了 {NFE:,} NFE，超過 1,000,000")
    fig, ax = _figure(n_seeds)
    for k, x in enumerate(xs):
        _plot_row(ax[k], x, f"Part {PROBLEM} · my method · seed {k}")
    plt.tight_layout(); plt.show()
    summ, per = P.summary(xs, _EXTRA[PROBLEM])
    ci = lambda a, b: "—" if a is None else f"{_f(a)} ～ {_f(b)}"
    print(f"Part {PROBLEM}：{n_seeds} 個 seed 的總表　NFE = {max(nfes):,}　mean R = {np.mean([reward(x).mean() for x in xs]):.2f}")
    tab = pd.DataFrame([(r["family"], r["name"], _f(r["mean"]), ci(*r["ci"]),
                         f"{r['range'][0]} ～ {r['range'][1]}" if r["key"] == "distinct" else f"{_f(r['range'][0])} ～ {_f(r['range'][1])}",
                         f"{r['k']}/{r['n']}" + (f"（{r['say']}）" if r["k"] and r["say"] else ""), _times(r["times"])) for r in summ],
                       columns=["HW1 的類別", "metric", "平均", "95% 信賴區間", "p_R 自己 20 次的範圍", "超出幾次", "差幾倍"])
    _show(tab, red=[r["k"] == r["n"] and r["judged"] for r in summ])   # N 次都超出才標紅；不同的點不標紅
    g = np.array([[v for _, v, _ in _groups(x)] for x in xs])
    print("　".join(f"{nm} 平均 {g[:, j].mean():.2f}（p_R 是 {ref:.2f}）" for j, (nm, _, ref) in enumerate(_groups(xs[0]))))
    table = pd.DataFrame([dict(seed=k, NFE=nfes[k], **{_SHORT.get(key, key): v for key, v in per[k].items()}) for k in range(n_seeds)])
    return table, xs

### 比較表的 metric：定義與說明

記號：你的 sample 是 $x_1,\dots,x_n$（$n = 2000$），參考點是 $y_1,\dots,y_m$，都來自 $p_R$：Part A 用 $p_R$ 的 4,000 個 quantile，Part B 用從 $p_R$ 抽的 2,000 個點。Part B 的投影平面用 `BASIS`（$2\times D$，兩列互相垂直、長度 1）表示，`x @ BASIS.T` 是點在平面上的座標；平面外指的是剩下的 $D-2$ 個方向。

**1. W1（Part A）、sliced W1（Part B）**　藍 搬運距離
- 定義：一維時 $W_1 = \int |F_x(t) - F_y(t)|\,dt$，$F_x$、$F_y$ 是兩組點的 empirical CDF。Part B 先把點投影到 128 個固定的隨機方向 $u_1,\dots,u_{128}$（長度 1），每個方向算一次一維的 $W_1$ 再平均：$\frac{1}{128}\sum_{j} W_1\big(u_j^\top x,\ u_j^\top y\big)$。
- 在量什麼：把你的點搬成參考點，平均要搬多遠。
- 哪一邊算超出：偏大。
- 看不到的東西：Part A 兩個峰相距 8.4，2,000 個點時兩峰的比例本身就會晃大約 ±1%，光是這樣 $W_1$ 就落在 0.01–0.15 之間，所以它主要反映兩峰的比例；峰的寬度就算錯到一半或兩倍、位置偏 0.1，大多看不出來。Part B 一個隨機方向落在投影平面上的部分平均只有 $2/D$，所以它主要反映平面外的方向，平面上的形狀幾乎看不出來。

**2. energy distance**　粉紅 成對距離
- 定義：$\mathcal{E} = 2\,\mathbb{E}\|X-Y\| - \mathbb{E}\|X-X'\| - \mathbb{E}\|Y-Y'\|$，$X, X'$ 是你的點，$Y, Y'$ 是參考點。組內的兩個平均不算點和自己的距離，所以兩組點來自同一個分布時，平均起來是 0，也可能是很小的負數。
- 在量什麼：兩組點之間的距離，比各自組內的距離大多少；只有兩個分布一樣時才是 0。
- 哪一邊算超出：偏大。它是平方等級的量，差幾倍用 $\sqrt{\mathcal{E}}$ 來比。
- 看不到的東西：和上面的 W1 一樣，Part A 主要反映兩峰的比例，Part B 主要反映平面外的方向。

**3. MMD²（RBF kernel）**　綠 密度與分格
- 定義：$\mathrm{MMD}^2 = \mathbb{E}\,k(X,X') + \mathbb{E}\,k(Y,Y') - 2\,\mathbb{E}\,k(X,Y)$，$k(a,b) = \exp\!\big(-\|a-b\|^2/(2h^2)\big)$；Part A 的 $h = 0.2$，Part B 的 $h = 2$。組內一樣不算點和自己，同一個分布時平均是 0。
- 在量什麼：用寬度 $h$ 的 kernel 把兩組點抹開成兩個密度，再比它們差多少；只有兩個分布一樣時才是 0。
- 哪一邊算超出：偏大。平方等級的量，差幾倍用 $\sqrt{\mathrm{MMD}^2}$ 來比。
- 看不到的東西：比 $h$ 小很多的差異。Part A 峰的寬度錯 30% 以上才穩定看得出來；Part B 和 energy distance 一樣主要反映平面外，平面上的形狀要錯很多才看得出來。

**4. 分格 TV**　綠 密度與分格
- 定義：把空間切成格子，再加一格放「格子外面」的點；$\hat p_b$、$\hat q_b$ 是你的點和參考點落在第 $b$ 格的比例，$\mathrm{TV} = \frac12\sum_b |\hat p_b - \hat q_b|$，在 0 到 1 之間。Part A 是 $[-9, 9]$ 上寬 0.1 的格子；Part B 是投影平面上 $[-4, 4]^2$ 的 16×16 格（每格 0.5×0.5）。
- 在量什麼：每一格的點數比例對不對。點數有限，就算真的從 $p_R$ 抽也不會是 0，所以要和範圍比。Part A 裡，它是看峰的寬度和位置最靈敏的一個。
- 哪一邊算超出：偏大。
- 看不到的東西：格子裡面的細節；Part B 只看投影平面，平面外的錯看不到。

**5. 各群比例差**　黃 數點
- 定義：每個點分到最近的中心（Part A：$-7, -4, -1, 1.5, 4, 7$；Part B：投影平面上四團的中心 $(\pm 2, 0)$、$(0, \pm 2)$），$\hat p_k$、$\hat q_k$ 是你的點和參考點分到第 $k$ 群的比例，比例差 $= \sum_k |\hat p_k - \hat q_k|$，在 0 到 2 之間。
- 在量什麼：各群的比例對不對（例如 Part A 兩個主要的峰是 76:24）。
- 哪一邊算超出：偏大。
- 看不到的東西：群裡面的形狀（寬窄、位置、重複的點）；Part B 只看投影平面。

**6. 覆蓋率（coverage，k = 5）**　黃 數點
- 定義：對每個參考點 $y_j$，$r_j$ 是它到第 5 近的參考點的距離；覆蓋率 $= \frac{1}{m}\sum_j \mathbf{1}\big[\min_i \|x_i - y_j\| \le r_j\big]$，也就是有多少比例的參考點，附近（$r_j$ 以內）至少有一個你的點（Naeem et al., 2020）。就算真的從 $p_R$ 抽，也只會是 0.93–0.98，不是 1。
- 在量什麼：參考點附近有沒有你的點。
- 哪一邊算超出：兩邊都算。偏小是有些地方沒蓋到（少了一個峰、很多重複的點、太散）；偏大是點比 $p_R$ 還集中。高維的時候，點擠在每一團的中心附近，反而會落進更多參考點的 $r_j$ 裡面，所以覆蓋率會比 $p_R$ 自己還高，不代表比較好。
- 看不到的東西：多出來的點（混進一些錯的點，覆蓋率幾乎不變）、各群比例差一點點。

**7. 各群寬度比**　橘 先分群再比
- 定義：用 5. 的分群，只看在參考點裡占超過 5% 的群。每一群算散開的程度 $s$：每個座標的變異數取平均再開根號。寬度比 $= \sum_g w_g\, s_{x,g} / s_{y,g}$，$w_g$ 是參考點裡第 $g$ 群占的比例（只算這幾群，加起來是 1）。1 是剛好。Part B 分成兩列：平面上（投影後的 2 個座標）和平面外（其餘 $D-2$ 個方向）。
- 在量什麼：每一群太寬還是太窄。
- 哪一邊算超出：兩邊都算，偏大是太散，偏小是太窄。
- 看不到的東西：各群的比例。另外它也會反映模型本身的誤差：不做 steering 的模型拿去和它自己的訓練資料比，Part A 偏寬 1–5%，Part B 平面外偏窄約 1%。Part B 平面外的範圍只有 ±0.5%，所以模型本身就會超出、差 1.5–4 倍，覆蓋率也常會稍微偏大；看這兩列要看差幾倍，不要只看有沒有標紅。

**8. 不同的點**
- 定義：2,000 個點四捨五入到小數 6 位之後，有幾個不一樣。
- 只當參考，不標紅：每個點加一點點雜訊（例如 0.001）就數不出重複，重複的問題主要看覆蓋率。一維時真的從 $p_R$ 抽也會有幾個點撞在一起，所以範圍是和 $p_R$ 自己 20 次裡最少的比。

**哪一邊算超出、差幾倍怎麼算**

| metric | 哪一邊算超出 | 差幾倍 |
|---|---|---|
| W1、sliced W1、分格 TV、各群比例差 | 偏大 | 值 ÷ 範圍上限 |
| energy distance、MMD² | 偏大 | √值 ÷ √範圍上限 |
| 覆蓋率 | 兩邊 | 偏小：(1 − 值) ÷ (1 − 範圍下限)；偏大：(1 − 範圍上限) ÷ (1 − 值) |
| 各群寬度比 | 兩邊 | \|log 值\| ÷ 同一側那一端的 \|log\|（偏窄和下限比，偏寬和上限比） |
| 自己加的 metric | 兩邊 | 離範圍中點的距離 ÷ 範圍的半寬 |
| 不同的點 | 不標紅 | — |

**要看什麼，看哪幾列**

| | Part A | Part B |
|---|---|---|
| 各群（峰）的比例 | 各群比例差、W1、energy distance | 各群比例差、分格 TV |
| 峰或團的寬度、位置 | 分格 TV、各群寬度比、MMD² | 分格 TV、各群寬度比：平面上 |
| 平面外的方向 | — | 各群寬度比：平面外、sliced W1、energy distance、MMD² |
| 漏掉一些地方、重複的點、點太集中 | 覆蓋率 | 覆蓋率 |

# Part A：Lab 的 1D 題目

Reward 和 lab 一樣：
$R(x)=2e^{-\frac{(x+7.2)^2}{2(0.3)^2}}+4e^{-\frac{(x+4.2)^2}{2(0.3)^2}}+3e^{-\frac{(x+1.3)^2}{2(0.3)^2}}+5e^{-\frac{(x-4.3)^2}{2(0.3)^2}}+e^{-\frac{(x-7.2)^2}{2(0.3)^2}}$，
$\lambda=3$，target 是 $p_R\propto p_{\text{model}}\,e^{\lambda R}$。

Lab 的時候看不到 $p_R$。這次 `compare(x)` 會把 $p_R$ 畫成黑線（用數值積分算出來的），和你的 sample 疊在一起，也會印出比較表和兩個主要的峰各占多少比例。先看看不做 steering 的樣子：

In [ ]:
use("A")
NFE = 0
_ = compare(sample(2000, steps=100), "Part A · no steering")

## （必做）你的方法

把你在 lab 最後交出去的方法（或你最滿意的那一版）貼進來。先不要改進它，這份作業想看的是當初的想法實際上做了什麼。如果 lab 的時候沒有寫出能跑的版本，就照當時的想法現在寫完，並在報告裡說明哪些是 lab 之後才補的。

為了讓同一段程式在 Part B 也能跑，貼過來之後請檢查：
1. 點的形狀寫成 `(n, D)`，例如 `torch.randn(n, 1)` 要改成 `torch.randn(n, D)`；
2. 在函式裡直接用 `LAM`、`reward`、`v_theta`、`sample`、`D`，不要把 3 或峰的位置寫死；
3. 回傳 `(n, D)` 的 numpy array 或 tensor。

<div style="background:rgba(255,183,0,.16); border-left:6px solid #e8a317; border-radius:4px; padding:8px 12px; margin-top:6px;"><b>要寫的地方</b>：下一格。把 lab 的 <code>my_method</code> 整段貼進去，取代裡面的 <code>raise NotImplementedError(...)</code>。整份 notebook 只有這一格一定要寫。</div>

In [ ]:
# ▼▼▼ 要寫的地方：把 lab 的 my_method 整段貼在這裡 ▼▼▼
def my_method(n=2000):
    raise NotImplementedError("還沒有把 lab 的方法貼進 my_method（往上找「要寫的地方」）")
# ▲▲▲ 寫到這裡為止 ▲▲▲

## （選做）把你在 HW1 A 設計的 metric 也加進比較表

寫成 `f(x, ref)`：`x` 是你的 sample、`ref` 是從 $p_R$ 抽的參考點，兩個都是 `(n, D)` 的 numpy array，回傳一個數字。用 `add_metric("名字", f)` 加進去之後，比較表會多一列，範圍一樣用 $p_R$ 自己的 20 組來算。Part B 的 `D` 比較大，如果你的 metric 只能用在 2D，可以先用 `x @ BASIS.T` 投影到 Part B 畫圖用的那個平面。

<div style="background:rgba(42,120,214,.10); border-left:6px solid #2a78d6; border-radius:4px; padding:8px 12px; margin-top:6px;"><b>選做</b>：寫在下一格。不寫的話直接 run 過去就好。</div>

In [ ]:
# ▼▼▼ 選做：把你在 HW1 A 設計的 metric 寫在這裡 ▼▼▼
# 例：
# def my_metric(x, ref):
#     return float(np.abs(x.mean(0) - ref.mean(0)).sum())
# add_metric("我的 metric（平均值的差）", my_metric)
# ▲▲▲ 寫到這裡為止 ▲▲▲

## 要跑幾個 seed

`run` 會用 `N_SEEDS` 個 random seed 各跑一次，總表的平均、信賴區間和「超出幾次」都跟著它。預設 5 個；想看得更穩可以加大，時間也會跟著變長。

<div style="background:rgba(42,120,214,.10); border-left:6px solid #2a78d6; border-radius:4px; padding:8px 12px; margin-top:6px;"><b>可以改</b>：下一格的 <code>N_SEEDS</code>（在 Colab 會顯示成拉桿）。不改就是 5 個。</div>

In [ ]:
N_SEEDS = 5  #@param {type:"slider", min:1, max:10, step:1}

In [ ]:
use("A")
table_A, xs_A = run(my_method)

# Part B：換一個模型

`use("B")` 會換成另一個已經 train 好的 flow matching 模型，資料是 D 維的向量（D 會印出來）。Reward 換成 $R(x) = -\dfrac{\lVert x - x^\star\rVert^2}{2\cdot 0.25}$，離 $x^\star$ 越近越好（$x^\star$ 存在 `XSTAR`）；$\lambda = 1$。Target 的形式一樣是 $p_R \propto p_{\text{model}}\,e^{\lambda R}$。

D 維的點沒辦法直接畫出來，所以 Part B 的 `compare(x)` 會畫四張圖：
- 前兩張：把點投影到一個固定的 2D 平面上，左邊是你的 sample，右邊是從 $p_R$ 抽出來的點（星號是 $x^\star$）；
- 第三張：reward $R(x)$ 的分布，你的 sample 和 $p_R$ 疊在一起；
- 第四張：每個點離那個平面有多遠（扣掉投影之後剩下的長度）。

比較表和 Part A 是同一張（metric 換成 D 維的版本），下面再加一行投影圖上兩團各占多少比例。

In [ ]:
use("B")
NFE = 0
_ = compare(sample(2000, steps=100), "Part B · no steering")

## 用同一個 `my_method` 跑 Part B

直接用同一個 `my_method`。如果它在 Part B 跑不動（例如用到只有 1D 才做得到的步驟），可以另外寫一個 `my_method_B`，想法不變、只改需要改的地方，然後把下一格的 `my_method` 換成 `my_method_B`。改了什麼、為什麼要改，請寫在報告裡。

<div style="background:rgba(42,120,214,.10); border-left:6px solid #2a78d6; border-radius:4px; padding:8px 12px; margin-top:6px;"><b>有需要才改</b>：下一格。<code>my_method</code> 在 Part B 能跑的話，直接 run 就好。</div>

In [ ]:
# 有需要才改：my_method 在 Part B 跑不動的話，把 my_method_B 寫在這裡，下面的 run(my_method) 換成 run(my_method_B)
use("B")
table_B, xs_B = run(my_method)

## （選做）其他實驗

如果想確認為什麼結果不符合預期，或是想試試報告最後一題的修改，可以寫在這裡，用 `run(...)` 或 `compare(...)` 畫出來。這部分不是必要的。

<div style="background:rgba(42,120,214,.10); border-left:6px solid #2a78d6; border-radius:4px; padding:8px 12px; margin-top:6px;"><b>選做</b>：寫在下一格。</div>

In [ ]:
# ▼▼▼ 選做：其他實驗寫在這裡 ▼▼▼

## 存檔

把 Part A、Part B 每個 seed 的 sample 存成 `hw2_samples.npz`，和 notebook、報告一起交。

In [ ]:
np.savez_compressed("hw2_samples.npz", A=np.stack(xs_A).astype(np.float32), B=np.stack(xs_B).astype(np.float32),
                    nfe_A=table_A.NFE.to_numpy(), nfe_B=table_B.NFE.to_numpy())
print("saved hw2_samples.npz", np.stack(xs_A).shape, np.stack(xs_B).shape)